# NeuroSAR Advanced — Full Research-Grade Pipeline
**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**

ResNet-50 Encoder · Dual-Pol VV+VH · Attention U-Net · Lovász Loss · EMA · TTA · GradCAM

## Cell 1 — Install

In [ ]:
import subprocess, sys
for pkg in ['torch>=2.0','torchvision','numpy','scipy','matplotlib',
            'scikit-learn','tqdm','Pillow','albumentations>=1.3.0',
            'timm>=0.9.0','grad-cam>=1.4.0']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
print('All packages ready')


## Cell 2 — Imports & Config

In [ ]:
import os,sys,json,time,random,warnings,zipfile,shutil
from pathlib import Path
from copy import deepcopy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from scipy.ndimage import gaussian_filter
import torch,torch.nn as nn,torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import GradScaler,autocast
import timm
from tqdm.notebook import tqdm
import albumentations as A
warnings.filterwarnings('ignore')

if torch.cuda.is_available():
    DEVICE=torch.device('cuda')
    p=torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  VRAM:{p.total_memory/1e9:.1f}GB')
else:
    DEVICE=torch.device('cpu'); print('CPU — set T4 GPU in runtime')

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark=True

DATA_DIR=Path('/content/neurosar_data'); EXP_DIR=Path('/content/neurosar_exp')
RESULTS_DIR=Path('/content/neurosar_results')
for d in [DATA_DIR,EXP_DIR,RESULTS_DIR]: d.mkdir(parents=True,exist_ok=True)

NUM_CLASSES=4
CLASS_NAMES=['background','vehicle','bunker','terrain_change']
CLASS_COLORS={0:'#1a1a2e',1:'#e03030',2:'#2064dc',3:'#f0b400'}
CLASS_CMAP=mcolors.ListedColormap([CLASS_COLORS[i] for i in range(NUM_CLASSES)])
CLASS_NORM=mcolors.BoundaryNorm(np.arange(-0.5,NUM_CLASSES+0.5),NUM_CLASSES)
CLASS_WEIGHTS=torch.tensor([0.15,2.0,1.8,1.5]).to(DEVICE)

CFG={'n_train':2000,'n_val':300,'n_test':300,'img_size':256,'in_channels':2,
     'encoder':'resnet50','base_filters':64,'deep_sup':True,'dropout':0.3,
     'epochs':80,'batch_size':8,'lr_max':3e-3,'weight_decay':1e-4,
     'patience':20,'ema_decay':0.9995,'mixup_alpha':0.4,'cutmix_alpha':0.4,'tta_folds':8}

print('Config ready | in_channels:',CFG['in_channels'],'(dual-pol VV+VH)')
print('Classes:',CLASS_NAMES)

## Cell 3 — Advanced SAR Data Generator
K-distribution speckle · dual-pol VV+VH · 5 terrain types · 4 military target sub-types

In [ ]:
def generate_speckle(shape,L=1,nu=4.0):
    texture=np.random.gamma(nu,1.0/nu,shape)
    speckle=np.zeros(shape)
    for _ in range(L):
        r=np.random.normal(0,1,shape); i=np.random.normal(0,1,shape)
        speckle+=r**2+i**2
    val=texture*(speckle/(2*L))
    return val/(val.mean()+1e-8)

def add_sar_noise(image,L=1,tdb=-25):
    sp=generate_speckle(image.shape,L=L)
    th=10**(tdb/10)*np.random.exponential(1.0,image.shape)
    return np.clip(image*sp+th,0,None)

def radio_cal(image):
    db=10*np.log10(np.clip(image,1e-6,None))
    return np.clip((db+30)/35.0,0,1).astype(np.float32)

def terrain_forest(s):
    b=np.random.exponential(0.35,s)
    for sc,am in [(2,.15),(5,.12),(15,.10),(40,.08)]: b+=gaussian_filter(np.random.normal(0,.1,s),sigma=sc)*am
    return np.clip(b,0.05,1.0)

def terrain_desert(s):
    b=gaussian_filter(np.random.exponential(0.08,s),sigma=12)
    x=np.linspace(0,8*np.pi,s[1]); y=np.linspace(0,8*np.pi,s[0]); X,Y=np.meshgrid(x,y)
    return np.clip(b+0.03*np.sin(X*0.5+np.random.uniform(0,np.pi))*np.cos(Y*0.3),0.01,0.5)

def terrain_urban(s):
    b=np.random.exponential(0.12,s); bs=np.random.randint(12,25)
    for r in range(0,s[0]-bs,bs+np.random.randint(4,10)):
        for c in range(0,s[1]-bs,bs+np.random.randint(4,10)):
            h=min(np.random.randint(8,bs),s[0]-r); w=min(np.random.randint(8,bs),s[1]-c)
            b[r:r+h,c:c+w]=np.random.uniform(0.55,0.95); b[r:r+2,c:c+w]=np.random.uniform(0.8,1.0)
    return np.clip(gaussian_filter(b,0.5),0.05,1.0)

def terrain_agricultural(s):
    b=np.full(s,0.1); fs=np.random.randint(25,55)
    vals={'wheat':0.25,'bare':0.08,'maize':0.35,'fallow':0.12}
    for i in range(0,s[0],fs):
        for j in range(0,s[1],fs): b[i:i+fs,j:j+fs]=vals[np.random.choice(list(vals.keys()))]
    for i in range(0,s[0],fs): b[max(0,i-1):i+2,:]=0.45
    for j in range(0,s[1],fs): b[:,max(0,j-1):j+2]=0.45
    return np.clip(b,0.02,0.8)

def terrain_wetland(s):
    b=np.random.exponential(0.18,s)
    for _ in range(np.random.randint(3,8)):
        r=np.random.randint(0,s[0]-20); c=np.random.randint(0,s[1]-20); rad=np.random.randint(8,25)
        Y,X=np.ogrid[-r:s[0]-r,-c:s[1]-c]; b[X**2+Y**2<rad**2]=np.random.uniform(0.01,0.05)
    return np.clip(gaussian_filter(b,2),0.01,0.7)

TERRAIN_GEN={'forest':terrain_forest,'desert':terrain_desert,'urban':terrain_urban,
             'agricultural':terrain_agricultural,'wetland':terrain_wetland}
LABELS={'background':0,'vehicle':1,'bunker':2,'terrain_change':3}

def render_vehicle(img,mask,row,col):
    h=np.random.randint(4,9); w=np.random.randint(5,12)
    r0,r1=max(0,row-h//2),min(img.shape[0],row+h//2+1)
    c0,c1=max(0,col-w//2),min(img.shape[1],col+w//2+1)
    ah,aw=r1-r0,c1-c0
    if ah<=0 or aw<=0: return
    img[r0:r1,c0:c1]=np.maximum(img[r0:r1,c0:c1],np.random.uniform(0.65,0.90,(ah,aw)))
    for dr,dc in [(0,0),(0,aw-1),(ah-1,0),(ah-1,aw-1)]:
        img[min(r0+dr,img.shape[0]-1),min(c0+dc,img.shape[1]-1)]=np.random.uniform(0.93,1.0)
    sl=np.random.randint(3,8)
    if r1+sl<=img.shape[0]: img[r1:r1+sl,c0:c1]*=np.random.uniform(0.02,0.12)
    mask[r0:r1,c0:c1]=LABELS['vehicle']

def render_bunker(img,mask,row,col):
    sz=np.random.randint(12,35)
    r0,r1=max(0,row-sz//2),min(img.shape[0],row+sz//2)
    c0,c1=max(0,col-sz//2),min(img.shape[1],col+sz//2)
    h,w=r1-r0,c1-c0
    if h<=3 or w<=3: return
    img[r0:r1,c0:c1]=np.random.uniform(0.50,0.75); mask[r0:r1,c0:c1]=LABELS['bunker']
    tk=max(2,sz//5)
    if r1-r0-2*tk>0 and c1-c0-2*tk>0: img[r0+tk:r1-tk,c0+tk:c1-tk]=np.random.uniform(0.04,0.18)
    if np.random.rand()<0.5: img[r0:r1,c0:c1]*=np.random.uniform(0.6,0.8)
    tl=np.random.randint(8,25); ctr=(c0+c1)//2
    img[max(0,r0-tl):r0,max(0,ctr-1):min(img.shape[1],ctr+2)]=np.random.uniform(0.28,0.48)

def render_change(img,mask,row,col):
    tp=np.random.choice(['crater','excavation','earthwork','minefield'],p=[0.3,0.3,0.25,0.15])
    if tp=='crater':
        rad=np.random.randint(6,24)
        Y,X=np.ogrid[-row:img.shape[0]-row,-col:img.shape[1]-col]; dist=np.sqrt(X**2+Y**2)
        rim=(dist>=rad*0.65)&(dist<=rad); interior=dist<rad*0.65
        img[rim]=np.random.uniform(0.55,0.85); mask[rim]=LABELS['terrain_change']
        img[interior]=np.random.uniform(0.01,0.08); mask[interior]=LABELS['terrain_change']
    elif tp=='excavation':
        h=np.random.randint(18,50); w=np.random.randint(20,70)
        r0,r1=max(0,row-h//2),min(img.shape[0],row+h//2)
        c0,c1=max(0,col-w//2),min(img.shape[1],col+w//2)
        img[r0:r1,c0:c1]=np.random.uniform(0.65,0.95); mask[r0:r1,c0:c1]=LABELS['terrain_change']
        if (r1-r0)>4 and (c1-c0)>4: img[r0+2:r1-2,c0+2:c1-2]=np.random.uniform(0.02,0.14)
    elif tp=='earthwork':
        ln=np.random.randint(20,60); wd=np.random.randint(4,10)
        ang=np.random.choice([0,3.14159/4,3.14159/2,3*3.14159/4])
        for t in range(-ln//2,ln//2):
            rr=int(row+t*np.cos(ang)); cc=int(col+t*np.sin(ang))
            for dw in range(-wd//2,wd//2):
                r2=np.clip(rr+int(dw*np.sin(ang)),0,img.shape[0]-1)
                c2=np.clip(cc+int(dw*np.cos(ang)),0,img.shape[1]-1)
                img[r2,c2]=np.random.uniform(0.55,0.80); mask[r2,c2]=LABELS['terrain_change']
    else:
        for _ in range(np.random.randint(5,15)):
            dr=np.random.randint(-30,30); dc=np.random.randint(-30,30)
            rr=np.clip(row+dr,2,img.shape[0]-3); cc=np.clip(col+dc,2,img.shape[1]-3)
            img[rr-1:rr+2,cc-1:cc+2]=np.random.uniform(0.45,0.70)
            mask[rr-1:rr+2,cc-1:cc+2]=LABELS['terrain_change']

RENDERERS={'vehicle':render_vehicle,'bunker':render_bunker,'terrain_change':render_change}

def compose_scene(image_size=256,dual_pol=True,num_targets_range=(1,8)):
    shape=(image_size,image_size)
    terrain=np.random.choice(list(TERRAIN_GEN.keys()))
    vv=TERRAIN_GEN[terrain](shape)
    vh=np.clip(vv*np.random.uniform(0.2,0.5)+terrain_forest(shape)*np.random.uniform(0.05,0.15),0,1)
    mask=np.zeros(shape,dtype=np.uint8); targets=[]
    for _ in range(np.random.randint(*num_targets_range)):
        row=np.random.randint(25,image_size-25); col=np.random.randint(25,image_size-25)
        tp=np.random.choice(['vehicle','bunker','terrain_change'],p=[0.4,0.3,0.3])
        RENDERERS[tp](vv,mask,row,col); RENDERERS[tp](vh,mask,row,col)
        targets.append({'type':tp,'row':int(row),'col':int(col)})
    L=np.random.choice([1,2,4])
    vv=radio_cal(add_sar_noise(vv,L=L)); vh=radio_cal(add_sar_noise(vh,L=L))
    img=(np.stack([vv,vh],axis=-1) if dual_pol else vv).astype(np.float32)
    return img,mask,{'terrain':terrain,'looks':L,'targets':targets,'n_targets':len(targets)}

def compose_change_pair(image_size=256,dual_pol=True):
    before,bmask,meta=compose_scene(image_size,dual_pol,num_targets_range=(0,3))
    after=before.copy(); amask=bmask.copy()
    for _ in range(np.random.randint(1,6)):
        row=np.random.randint(25,image_size-25); col=np.random.randint(25,image_size-25)
        tp=np.random.choice(['vehicle','bunker','terrain_change'],p=[0.5,0.25,0.25])
        if dual_pol: RENDERERS[tp](after[:,:,0],amask,row,col); RENDERERS[tp](after[:,:,1],amask,row,col)
        else: RENDERERS[tp](after,amask,row,col)
    L=np.random.choice([1,2,4])
    if dual_pol:
        after[:,:,0]=radio_cal(add_sar_noise(after[:,:,0],L=L))
        after[:,:,1]=radio_cal(add_sar_noise(after[:,:,1],L=L))
    else: after=radio_cal(add_sar_noise(after,L=L))
    return before.astype(np.float32),after.astype(np.float32),(amask!=bmask).astype(np.uint8)

print('SAR generator ready')
print('  Speckle: K-distribution (physically accurate)')
print('  Terrain: forest/desert/urban/agricultural/wetland')
print('  Targets: vehicle/bunker/crater/excavation/earthwork/minefield')
print('  Dual-pol: VV + VH channels')

## Cell 4 — Preview Dual-Pol SAR Scenes

In [ ]:
np.random.seed(7)
terrains=list(TERRAIN_GEN.keys())
fig,axes=plt.subplots(3,5,figsize=(22,13),facecolor='#06090f')
fig.suptitle('NeuroSAR Advanced — Dual-Pol SAR Preview',color='white',fontsize=14,fontweight='bold')
for ci,terrain in enumerate(terrains):
    img,mask,meta=compose_scene(256,dual_pol=True,num_targets_range=(2,6))
    vv=img[:,:,0]; vh=img[:,:,1]
    ov=np.zeros((*mask.shape,4))
    for c in range(1,4):
        if (mask==c).any():
            rgb=mcolors.to_rgb(CLASS_COLORS[c]); ov[mask==c]=(*rgb,0.65)
    for ri,(data,ttl) in enumerate([(vv,'VV pol'),(vh,'VH pol'),(np.stack([vv,vh,np.clip(vv-vh,0,1)],axis=-1),'RGB composite')]):
        ax=axes[ri,ci]; ax.set_facecolor('#0d1520')
        if ri<2: ax.imshow(data,cmap='gray',vmin=0,vmax=1); ax.imshow(ov)
        else: ax.imshow(data)
        ax.set_title(f'{terrain} {ttl}',color='#aaccff',fontsize=8); ax.axis('off')
plt.tight_layout()
plt.savefig(RESULTS_DIR/'01_dual_pol_preview.png',dpi=120,bbox_inches='tight',facecolor='#06090f')
plt.show(); print('Saved 01_dual_pol_preview.png')

## Cell 5 — Generate Full Dataset

In [ ]:
def build_dataset(output_dir,n_train,n_val,n_test,image_size=256,dual_pol=True):
    output_dir=Path(output_dir)
    for split,n in [('train',n_train),('val',n_val),('test',n_test)]:
        sd=output_dir/split
        for sub in ['images','masks','before','after','change_masks']: (sd/sub).mkdir(parents=True,exist_ok=True)
        manifest=[]
        for i in tqdm(range(n),desc=f'{split:5s}',leave=False):
            img,mask,meta=compose_scene(image_size,dual_pol=dual_pol)
            np.save(sd/'images'/f'{i:05d}.npy',img.astype(np.float32))
            np.save(sd/'masks' /f'{i:05d}.npy',mask)
            bef,aft,cm=compose_change_pair(image_size,dual_pol=dual_pol)
            np.save(sd/'before'      /f'{i:05d}.npy',bef)
            np.save(sd/'after'       /f'{i:05d}.npy',aft)
            np.save(sd/'change_masks'/f'{i:05d}.npy',cm)
            manifest.append({'id':i,**meta})
        with open(sd/'manifest.json','w') as f:
            json.dump(manifest,f,indent=2,default=lambda x:int(x) if hasattr(x,'item') else str(x))
        print(f'  {split}: {n} samples saved')
    with open(output_dir/'class_info.json','w') as f:
        json.dump({'classes':LABELS,'num_classes':NUM_CLASSES,'dual_pol':dual_pol},f,indent=2)

np.random.seed(SEED); t0=time.time()
dual=(CFG['in_channels']==2)
build_dataset(DATA_DIR,CFG['n_train'],CFG['n_val'],CFG['n_test'],CFG['img_size'],dual_pol=dual)
print(f'Dataset complete in {time.time()-t0:.0f}s')

## Cell 6 — Advanced Augmentation & Dataset Classes
MixUp + CutMix + Albumentations

In [ ]:
def get_aug():
    return A.Compose([
        A.HorizontalFlip(p=0.5),A.VerticalFlip(p=0.3),A.RandomRotate90(p=0.4),A.Transpose(p=0.2),
        A.ShiftScaleRotate(shift_limit=0.05,scale_limit=0.1,rotate_limit=15,border_mode=0,p=0.4),
        A.ElasticTransform(alpha=30,sigma=5,p=0.2),
        A.GridDistortion(num_steps=5,distort_limit=0.2,p=0.2),
        A.CoarseDropout(max_holes=4,max_height=32,max_width=32,p=0.2),
        A.RandomGamma(gamma_limit=(70,130),p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.15,contrast_limit=0.15,p=0.3),
    ])

def mixup(i1,m1,i2,m2,a=0.4):
    lam=np.random.beta(a,a)
    return (lam*i1+(1-lam)*i2).astype(np.float32),np.where(np.random.rand(*m1.shape)<lam,m1,m2).astype(np.int64)

def cutmix(i1,m1,i2,m2,a=0.4):
    H,W=i1.shape[:2]; lam=np.random.beta(a,a)
    ch=int(H*np.sqrt(1-lam)); cw=int(W*np.sqrt(1-lam))
    r=np.random.randint(0,H-ch+1); c=np.random.randint(0,W-cw+1)
    ri=i1.copy(); rm=m1.copy()
    ri[r:r+ch,c:c+cw]=i2[r:r+ch,c:c+cw]; rm[r:r+ch,c:c+cw]=m2[r:r+ch,c:c+cw]
    return ri.astype(np.float32),rm.astype(np.int64)

class SARSegDataset(Dataset):
    def __init__(self,root,split='train',augment=True,in_channels=2,mixup_a=0.4,cutmix_a=0.4):
        self.root=Path(root)/split; self.augment=augment and (split=='train')
        self.in_ch=in_channels; self.ma=mixup_a; self.ca=cutmix_a
        self.tfm=get_aug() if self.augment else None
        self.ids=sorted([p.stem for p in (self.root/'images').glob('*.npy')])
        if not self.ids: raise FileNotFoundError(f'No data in {self.root}')
    def __len__(self): return len(self.ids)
    def _load(self,idx):
        id_=self.ids[idx]
        img=np.load(self.root/'images'/f'{id_}.npy').astype(np.float32)
        msk=np.load(self.root/'masks' /f'{id_}.npy').astype(np.int64)
        if img.ndim==2: img=img[:,:,np.newaxis]
        if self.in_ch==1 and img.shape[2]>1: img=img[:,:,:1]
        return img,msk
    def __getitem__(self,idx):
        img,msk=self._load(idx)
        if self.augment and self.tfm:
            a=self.tfm(image=img,mask=msk); img,msk=a['image'],a['mask']
        if self.augment:
            r=np.random.rand()
            if r<0.3:
                i2,m2=self._load(np.random.randint(len(self)))
                if self.tfm: a2=self.tfm(image=i2,mask=m2); i2,m2=a2['image'],a2['mask']
                img,msk=mixup(img,msk,i2,m2,self.ma)
            elif r<0.6:
                i2,m2=self._load(np.random.randint(len(self)))
                if self.tfm: a2=self.tfm(image=i2,mask=m2); i2,m2=a2['image'],a2['mask']
                img,msk=cutmix(img,msk,i2,m2,self.ca)
        return {'image':torch.from_numpy(img.transpose(2,0,1)),
                'mask':torch.from_numpy(msk.astype(np.int64)),'id':self.ids[idx]}

class SARChangeDataset(Dataset):
    def __init__(self,root,split='train',augment=True,in_channels=2):
        self.root=Path(root)/split; self.augment=augment and (split=='train'); self.in_ch=in_channels
        self.ids=sorted([p.stem for p in (self.root/'before').glob('*.npy')])
    def __len__(self): return len(self.ids)
    def __getitem__(self,idx):
        id_=self.ids[idx]
        bef=np.load(self.root/'before'      /f'{id_}.npy').astype(np.float32)
        aft=np.load(self.root/'after'       /f'{id_}.npy').astype(np.float32)
        chg=np.load(self.root/'change_masks'/f'{id_}.npy').astype(np.float32)
        if bef.ndim==2: bef=bef[:,:,np.newaxis]; aft=aft[:,:,np.newaxis]
        if self.in_ch==1: bef=bef[:,:,:1]; aft=aft[:,:,:1]
        if self.augment and np.random.rand()<0.5:
            bef=np.fliplr(bef).copy(); aft=np.fliplr(aft).copy(); chg=np.fliplr(chg).copy()
        return {'before':torch.from_numpy(bef.transpose(2,0,1)),
                'after': torch.from_numpy(aft.transpose(2,0,1)),
                'change':torch.from_numpy(chg[np.newaxis]),'id':id_}

def get_loaders(data_root,cfg,task='segmentation'):
    kw={'root':data_root,'in_channels':cfg['in_channels']}
    if task=='segmentation':
        tr=SARSegDataset(split='train',augment=True,
                          mixup_a=cfg.get('mixup_alpha',0.4),cutmix_a=cfg.get('cutmix_alpha',0.4),**kw)
        DS=SARSegDataset
    else:
        tr=SARChangeDataset(split='train',augment=True,**kw); DS=SARChangeDataset
    va=DS(split='val',augment=False,**kw); te=DS(split='test',augment=False,**kw)
    mk=lambda ds,sh: DataLoader(ds,batch_size=cfg['batch_size'],shuffle=sh,num_workers=2,pin_memory=True,drop_last=sh)
    return mk(tr,True),mk(va,False),mk(te,False)

train_loader,val_loader,test_loader=get_loaders(DATA_DIR,CFG,'segmentation')
b=next(iter(train_loader))
print(f'DataLoader OK  image:{list(b["image"].shape)}  mask:{list(b["mask"].shape)}')
print(f'Unique labels: {b["mask"].unique().tolist()}')

## Cell 7 — Advanced Model
ResNet-50 pretrained encoder + Attention gates + SE blocks + ASPP bottleneck + EMA

In [ ]:
class ConvBNAct(nn.Module):
    def __init__(self,in_ch,out_ch,kernel=3,padding=1,dilation=1,act=True,dropout=0.0):
        super().__init__()
        gn=min(8,out_ch)
        while out_ch%gn!=0 and gn>1: gn-=1
        layers=[nn.Conv2d(in_ch,out_ch,kernel,padding=padding,dilation=dilation,bias=False),nn.GroupNorm(gn,out_ch)]
        if act: layers.append(nn.GELU())
        if dropout>0: layers.append(nn.Dropout2d(dropout))
        self.block=nn.Sequential(*layers)
    def forward(self,x): return self.block(x)

class SE(nn.Module):
    def __init__(self,ch,r=16):
        super().__init__()
        self.pool=nn.AdaptiveAvgPool2d(1)
        self.fc=nn.Sequential(nn.Linear(ch,max(1,ch//r)),nn.GELU(),nn.Linear(max(1,ch//r),ch),nn.Sigmoid())
    def forward(self,x):
        b,c,_,_=x.shape; return x*self.fc(self.pool(x).view(b,c)).view(b,c,1,1)

class AttentionGate(nn.Module):
    def __init__(self,Fg,Fl,Fi):
        super().__init__()
        Fi=max(1,Fi); gn=max(1,min(8,Fi))
        while Fi%gn!=0 and gn>1: gn-=1
        self.Wg=nn.Sequential(nn.Conv2d(Fg,Fi,1,bias=True),nn.GroupNorm(gn,Fi))
        self.Wx=nn.Sequential(nn.Conv2d(Fl,Fi,1,bias=True),nn.GroupNorm(gn,Fi))
        self.psi=nn.Sequential(nn.Conv2d(Fi,1,1,bias=True),nn.GroupNorm(1,1),nn.Sigmoid())
        self.act=nn.GELU()
    def forward(self,g,x):
        g1=self.Wg(g); x1=self.Wx(x)
        if g1.shape[2:]!=x1.shape[2:]: g1=F.interpolate(g1,size=x1.shape[2:],mode='bilinear',align_corners=False)
        return x*self.psi(self.act(g1+x1))

class ASPP(nn.Module):
    def __init__(self,in_ch,out_ch,rates=(1,6,12,18,24)):
        super().__init__()
        self.c1=ConvBNAct(in_ch,out_ch,1,padding=0)
        self.branches=nn.ModuleList()
        for r in rates[1:]:
            self.branches.append(nn.Sequential(
                nn.Conv2d(in_ch,in_ch,3,padding=r,dilation=r,groups=in_ch,bias=False),
                nn.Conv2d(in_ch,out_ch,1,bias=False),
                nn.GroupNorm(max(1,min(8,out_ch)),out_ch),nn.GELU()))
        self.gap=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(in_ch,out_ch,1,bias=False),nn.GELU())
        self.proj=ConvBNAct(out_ch*(len(rates)+1),out_ch,1,padding=0)
    def forward(self,x):
        H,W=x.shape[2:]; feats=[self.c1(x)]+[b(x) for b in self.branches]
        feats.append(F.interpolate(self.gap(x),(H,W),mode='bilinear',align_corners=False))
        return self.proj(torch.cat(feats,1))

class DecBlock(nn.Module):
    def __init__(self,in_ch,skip_ch,out_ch,nc=None,dropout=0.0):
        super().__init__()
        self.up=nn.ConvTranspose2d(in_ch,in_ch//2,2,stride=2)
        self.att=AttentionGate(in_ch//2,skip_ch,skip_ch//2)
        self.se=SE(in_ch//2+skip_ch)
        self.conv=nn.Sequential(ConvBNAct(in_ch//2+skip_ch,out_ch,dropout=dropout),ConvBNAct(out_ch,out_ch,dropout=dropout))
        self.aux=nn.Conv2d(out_ch,nc,1) if nc else None
    def forward(self,x,skip,img_size=None):
        x=self.up(x); sa=self.att(x,skip); x=self.se(torch.cat([x,sa],1)); x=self.conv(x)
        aux=F.interpolate(self.aux(x),size=img_size,mode='bilinear',align_corners=False) if self.aux and img_size else None
        return x,aux

class NeuroSAR_UNet(nn.Module):
    def __init__(self,in_channels=2,num_classes=4,encoder_name='resnet50',pretrained=True,deep_sup=True,dropout=0.3):
        super().__init__()
        self.ds=deep_sup
        self.enc=timm.create_model(encoder_name,pretrained=pretrained,features_only=True,out_indices=(0,1,2,3,4),in_chans=in_channels)
        ec=self.enc.feature_info.channels()
        self.aspp=ASPP(ec[4],512)
        nc=num_classes if deep_sup else None
        self.d4=DecBlock(512,ec[3],256,nc,dropout); self.d3=DecBlock(256,ec[2],128,nc,dropout)
        self.d2=DecBlock(128,ec[1],64,nc,dropout); self.d1=DecBlock(64,ec[0],32,None,dropout)
        self.head=nn.Sequential(ConvBNAct(32,32,dropout=dropout),nn.Conv2d(32,num_classes,1))
        self._init()
    def _init(self):
        for n,m in self.named_modules():
            if 'enc' in n: continue
            if isinstance(m,nn.Conv2d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m,(nn.GroupNorm,nn.BatchNorm2d)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    def forward(self,x):
        sz=x.shape[2:]; s0,s1,s2,s3,s4=self.enc(x); xb=self.aspp(s4)
        d4,a4=self.d4(xb,s3,sz); d3,a3=self.d3(d4,s2,sz); d2,a2=self.d2(d3,s1,sz); d1,_=self.d1(d2,s0)
        out=self.head(d1)
        if out.shape[2:]!=sz: out=F.interpolate(out,sz,mode='bilinear',align_corners=False)
        res={'logits':out}
        if self.ds and self.training: res.update({'aux4':a4,'aux3':a3,'aux2':a2})
        return res

class EMA:
    def __init__(self,model,decay=0.9995):
        self.decay=decay; self.shadow=deepcopy(model); self.shadow.eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self,model):
        for (_,sp),(_,mp) in zip(self.shadow.named_parameters(),model.named_parameters()):
            sp.data.mul_(self.decay).add_(mp.data,alpha=1-self.decay)

model=NeuroSAR_UNet(in_channels=CFG['in_channels'],num_classes=NUM_CLASSES,
                     encoder_name=CFG['encoder'],pretrained=True,deep_sup=True,dropout=CFG['dropout']).to(DEVICE)
x=torch.randn(2,CFG['in_channels'],256,256).to(DEVICE); model.train(); out=model(x)
total=sum(p.numel() for p in model.parameters())
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'NeuroSAR U-Net — ResNet-50 pretrained + Attention + SE + ASPP + DeepSup')
print(f'Output: {list(out["logits"].shape)}')
print(f'Params: {total/1e6:.1f}M total | {trainable/1e6:.1f}M trainable')
print(f'Aux keys: {[k for k in out if "aux" in k]}')

## Cell 8 — Research-Grade Losses & Metrics
Focal + Lovász-Softmax (optimises IoU directly) + Dice + Deep Supervision

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self,alpha=None,gamma=2.5): super().__init__(); self.alpha=alpha; self.gamma=gamma
    def forward(self,logits,targets):
        lp=F.log_softmax(logits,1); lpt=lp.gather(1,targets.unsqueeze(1).clamp(0,logits.shape[1]-1)); pt=lpt.exp()
        at=self.alpha[targets.clamp(0)].unsqueeze(1) if self.alpha is not None else 1.0
        return (-at*(1-pt)**self.gamma*lpt).mean()

class LovaszLoss(nn.Module):
    def forward(self,logits,targets):
        probs=F.softmax(logits,1); B,C,H,W=probs.shape; loss=[]
        for c in range(C):
            fg=(targets==c).float()
            if fg.sum()==0: continue
            errors=(fg-probs[:,c]).abs(); es,perm=torch.sort(errors.view(-1),descending=True)
            fg_s=fg.view(-1)[perm]; gts=fg_s.sum()
            inter=gts-fg_s.cumsum(0); union=gts+(1-fg_s).cumsum(0)
            iou=1.0-inter/union.clamp(min=1); iou[1:]=iou[1:]-iou[:-1]
            loss.append(torch.dot(es,iou))
        return torch.stack(loss).mean() if loss else probs.sum()*0

class DiceLoss(nn.Module):
    def __init__(self,s=1.0): super().__init__(); self.s=s
    def forward(self,logits,targets):
        p=F.softmax(logits,1); C=logits.shape[1]; B=logits.shape[0]
        t_oh=F.one_hot(targets,C).permute(0,3,1,2).float(); losses=[]
        for c in range(1,C):
            pc=p[:,c].reshape(B,-1); tc=t_oh[:,c].reshape(B,-1)
            losses.append(1-(2*(pc*tc).sum(1)+self.s)/(pc.sum(1)+tc.sum(1)+self.s))
        return torch.stack(losses).mean()

class AdvSegLoss(nn.Module):
    def __init__(self,num_classes=4,class_weights=None):
        super().__init__()
        self.focal=FocalLoss(alpha=class_weights,gamma=2.5); self.lovasz=LovaszLoss(); self.dice=DiceLoss()
    def forward(self,outputs,targets):
        logits=outputs['logits']
        main=0.3*self.focal(logits,targets)+0.5*self.lovasz(logits,targets)+0.2*self.dice(logits,targets)
        deep=torch.tensor(0.0,device=logits.device)
        ak=[k for k in outputs if k.startswith('aux') and outputs[k] is not None]
        if ak:
            for k in ak: deep+=0.3*self.focal(outputs[k],targets)+0.4*self.lovasz(outputs[k],targets)+0.3*self.dice(outputs[k],targets)
            deep/=len(ak)
        return main+0.4*deep

def bfd_loss(logits,targets,gamma=2.0,pw=3.0,smooth=1.0):
    logits=logits.squeeze(1); targets=targets.squeeze(1).float(); prob=torch.sigmoid(logits)
    bce=F.binary_cross_entropy_with_logits(logits,targets,pos_weight=torch.tensor(pw,device=logits.device),reduction='none')
    pt=torch.where(targets==1,prob,1-prob); focal=((1-pt)**gamma*bce).mean()
    p=prob.reshape(-1); t=targets.reshape(-1)
    return 0.5*focal+0.5*(1-(2*(p*t).sum()+smooth)/(p.sum()+t.sum()+smooth))

class SegMetrics:
    def __init__(self,nc=4): self.nc=nc; self.reset()
    def reset(self): self.C=np.zeros((self.nc,self.nc),dtype=np.int64)
    def update(self,preds,targets):
        p=preds.cpu().numpy().flatten() if isinstance(preds,torch.Tensor) else preds.flatten()
        t=targets.cpu().numpy().flatten() if isinstance(targets,torch.Tensor) else targets.flatten()
        v=(t>=0)&(t<self.nc); p,t=p[v],t[v]
        self.C+=np.bincount(t*self.nc+p,minlength=self.nc**2).reshape(self.nc,self.nc)
    def compute(self):
        C=self.C; il,fl=[],[]; res={}
        for c in range(self.nc):
            TP=C[c,c]; FP=C[:,c].sum()-TP; FN=C[c,:].sum()-TP
            iou=TP/(TP+FP+FN+1e-8); pr=TP/(TP+FP+1e-8); rc=TP/(TP+FN+1e-8); f1=2*pr*rc/(pr+rc+1e-8)
            nm=CLASS_NAMES[c] if c<len(CLASS_NAMES) else f'cls{c}'
            res[f'iou_{nm}']=float(iou); res[f'f1_{nm}']=float(f1); res[f'prec_{nm}']=float(pr); res[f'rec_{nm}']=float(rc)
            il.append(iou); fl.append(f1)
        res['mIoU']=float(np.mean(il)); res['mIoU_targets']=float(np.mean(il[1:]))
        res['mF1_targets']=float(np.mean(fl[1:])); res['pixel_acc']=float(np.diag(C).sum()/(C.sum()+1e-8))
        return res

class CDMetrics:
    def __init__(self): self.reset()
    def reset(self): self.TP=self.TN=self.FP=self.FN=0
    def update(self,logits,targets):
        p=(torch.sigmoid(logits.squeeze(1))>0.5).long().cpu().numpy().flatten()
        t=targets.squeeze(1).long().cpu().numpy().flatten()
        self.TP+=int(((p==1)&(t==1)).sum()); self.TN+=int(((p==0)&(t==0)).sum())
        self.FP+=int(((p==1)&(t==0)).sum()); self.FN+=int(((p==0)&(t==1)).sum())
    def compute(self):
        TP,TN,FP,FN=self.TP,self.TN,self.FP,self.FN
        pr=TP/(TP+FP+1e-8); rc=TP/(TP+FN+1e-8); f1=2*pr*rc/(pr+rc+1e-8)
        return {'f1':float(f1),'iou':float(TP/(TP+FP+FN+1e-8)),'far':float(FP/(FP+TN+1e-8)),
                'mar':float(FN/(FN+TP+1e-8)),'precision':float(pr),'recall':float(rc)}

print('Losses: FocalLoss + LovaszLoss (direct IoU) + DiceLoss + DeepSupervision')
print('Metrics: SegMetrics + CDMetrics')

## Cell 9 — Train Segmentation Model
OneCycleLR superconvergence · differential LRs · EMA · mixed precision

In [ ]:
model=NeuroSAR_UNet(in_channels=CFG['in_channels'],num_classes=NUM_CLASSES,
                     encoder_name=CFG['encoder'],pretrained=True,deep_sup=True,dropout=CFG['dropout']).to(DEVICE)
criterion=AdvSegLoss(num_classes=NUM_CLASSES,class_weights=CLASS_WEIGHTS)
ema=EMA(model,decay=CFG['ema_decay'])

enc_p=[p for n,p in model.named_parameters() if 'enc' in n and p.requires_grad]
dec_p=[p for n,p in model.named_parameters() if 'enc' not in n]
optimizer=optim.AdamW([{'params':enc_p,'lr':CFG['lr_max']/10},{'params':dec_p,'lr':CFG['lr_max']}],weight_decay=CFG['weight_decay'])
train_loader,val_loader,test_loader=get_loaders(DATA_DIR,CFG,'segmentation')
scheduler=OneCycleLR(optimizer,max_lr=[CFG['lr_max']/10,CFG['lr_max']],
                      steps_per_epoch=len(train_loader),epochs=CFG['epochs'],
                      pct_start=0.1,div_factor=25,final_div_factor=1e4,anneal_strategy='cos')
scaler=GradScaler(enabled=(DEVICE.type=='cuda'))

total=sum(p.numel() for p in model.parameters())
print(f'Model: {total/1e6:.1f}M params | {len(train_loader)} batches/epoch')

history={'train_loss':[],'val_loss':[],'val_mIoU':[],'val_mIoU_tgt':[],'lr':[]}
best_score=0.0; patience_ctr=0
best_path=EXP_DIR/'best_model.pth'; ema_path=EXP_DIR/'ema_model.pth'

for epoch in range(1,CFG['epochs']+1):
    model.train(); tr_loss=0.0; tr_m=SegMetrics(NUM_CLASSES)
    bar=tqdm(train_loader,desc=f'Ep {epoch:3d}/{CFG["epochs"]}',leave=False)
    for batch in bar:
        images=batch['image'].to(DEVICE); targets=batch['mask'].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(DEVICE.type=='cuda')):
            out=model(images); loss=criterion(out,targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step(); ema.update(model)
        tr_loss+=loss.item(); tr_m.update(out['logits'].argmax(1),targets)
        bar.set_postfix(loss=f'{loss.item():.4f}')
    ema.shadow.eval(); va_loss=0.0; va_m=SegMetrics(NUM_CLASSES)
    with torch.no_grad():
        for batch in val_loader:
            images=batch['image'].to(DEVICE); targets=batch['mask'].to(DEVICE)
            with autocast(enabled=(DEVICE.type=='cuda')):
                out=ema.shadow(images); loss=criterion(out,targets)
            va_loss+=loss.item(); va_m.update(out['logits'].argmax(1),targets)
    vr=va_m.compute(); lr=scheduler.get_last_lr()[-1]
    history['train_loss'].append(tr_loss/len(train_loader)); history['val_loss'].append(va_loss/len(val_loader))
    history['val_mIoU'].append(vr['mIoU']); history['val_mIoU_tgt'].append(vr['mIoU_targets']); history['lr'].append(lr)
    star=''
    if vr['mIoU_targets']>best_score:
        best_score=vr['mIoU_targets']; patience_ctr=0
        torch.save({'epoch':epoch,'state':model.state_dict(),'score':best_score,'cfg':CFG},best_path)
        torch.save({'epoch':epoch,'state':ema.shadow.state_dict(),'score':best_score},ema_path); star=' BEST'
    else: patience_ctr+=1
    print(f'Ep {epoch:3d} | TrL={tr_loss/len(train_loader):.4f} VaL={va_loss/len(val_loader):.4f} | mIoU={vr["mIoU"]:.4f} mIoU_tgt={vr["mIoU_targets"]:.4f} | LR={lr:.2e}{star}')
    if patience_ctr>=CFG['patience']: print(f'Early stop at epoch {epoch}'); break

print(f'Training complete! Best mIoU_targets = {best_score:.4f}')

## Cell 10 — Training Curves

In [ ]:
ep=list(range(1,len(history['train_loss'])+1))
fig,axes=plt.subplots(1,3,figsize=(16,4),facecolor='#0d0d0d')
def sty(ax,ti,yl):
    ax.set_facecolor('#1a1a1a'); ax.set_title(ti,color='#ccc',fontsize=10)
    ax.set_xlabel('Epoch',color='#888',fontsize=8); ax.set_ylabel(yl,color='#888',fontsize=8)
    ax.tick_params(colors='#888',labelsize=7); ax.grid(True,color='#2a2a2a',lw=0.8)
    for sp in ax.spines.values(): sp.set_edgecolor('#333')
    try: ax.legend(facecolor='#1a1a1a',edgecolor='#444',labelcolor='white',fontsize=8)
    except: pass
axes[0].plot(ep,history['train_loss'],'#5EB8FF',lw=2,label='Train')
axes[0].plot(ep,history['val_loss'],'#FF6B6B',lw=2,label='Val EMA'); sty(axes[0],'Loss','Loss')
axes[1].plot(ep,history['val_mIoU'],'#64FF96',lw=2,label='mIoU all')
axes[1].plot(ep,history['val_mIoU_tgt'],'#FFB347',lw=2,label='mIoU targets'); sty(axes[1],'mIoU','mIoU')
axes[2].semilogy(ep,history['lr'],'#b060ff',lw=2)
axes[2].fill_between(ep,min(history['lr']),history['lr'],color='#b060ff',alpha=0.15); sty(axes[2],'OneCycleLR','LR')
plt.tight_layout()
plt.savefig(RESULTS_DIR/'02_training_curves.png',dpi=130,bbox_inches='tight',facecolor='#0d0d0d')
plt.show(); print('Saved 02_training_curves.png')

## Cell 11 — Test Evaluation & Confusion Matrix

In [ ]:
ckpt=torch.load(ema_path,map_location=DEVICE)
ema.shadow.load_state_dict(ckpt['state']); ema.shadow.eval()
print(f'Loaded EMA model — epoch {ckpt["epoch"]}  score={ckpt["score"]:.4f}')

test_m=SegMetrics(NUM_CLASSES); conf_m=np.zeros((NUM_CLASSES,NUM_CLASSES),dtype=np.int64)
with torch.no_grad():
    for batch in tqdm(test_loader,desc='Testing'):
        images=batch['image'].to(DEVICE); targets=batch['mask'].to(DEVICE)
        with autocast(enabled=(DEVICE.type=='cuda')): out=ema.shadow(images)
        preds=out['logits'].argmax(1); test_m.update(preds,targets)
        p=preds.cpu().numpy().flatten(); t=targets.cpu().numpy().flatten()
        conf_m+=np.bincount(t*NUM_CLASSES+p,minlength=NUM_CLASSES**2).reshape(NUM_CLASSES,NUM_CLASSES)

results=test_m.compute()
print('\n====== TEST RESULTS ======')
print(f'  Pixel Accuracy : {results["pixel_acc"]:.4f}')
print(f'  mIoU (all)     : {results["mIoU"]:.4f}')
print(f'  mIoU (targets) : {results["mIoU_targets"]:.4f}')
print(f'  mF1  (targets) : {results["mF1_targets"]:.4f}')
for name in CLASS_NAMES[1:]:
    print(f'  {name:20s}  IoU={results[f"iou_{name}"]:.4f}  F1={results[f"f1_{name}"]:.4f}  P={results[f"prec_{name}"]:.4f}  R={results[f"rec_{name}"]:.4f}')

fig,ax=plt.subplots(figsize=(7,6),facecolor='#0d0d0d'); ax.set_facecolor('#1a1a1a')
cm_n=conf_m.astype(float)/(conf_m.sum(1,keepdims=True)+1e-8)
im=ax.imshow(cm_n,cmap='Blues',vmin=0,vmax=1); plt.colorbar(im,ax=ax,fraction=0.046,pad=0.04)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j,i,f'{cm_n[i,j]:.2f}\n({conf_m[i,j]})',ha='center',va='center',fontsize=8,
                color='white' if cm_n[i,j]<0.5 else 'black')
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES,color='#ccc',rotation=30,ha='right'); ax.set_yticklabels(CLASS_NAMES,color='#ccc')
ax.set_xlabel('Predicted',color='#999'); ax.set_ylabel('True',color='#999')
ax.set_title('Confusion Matrix — EMA model',color='white',fontsize=12); ax.tick_params(colors='#888')
plt.tight_layout()
plt.savefig(RESULTS_DIR/'03_confusion_matrix.png',dpi=130,bbox_inches='tight',facecolor='#0d0d0d')
plt.show()

## Cell 12 — Visual Predictions on Test Samples

In [ ]:
ema.shadow.eval(); test_ds=test_loader.dataset
N=6; indices=np.random.choice(len(test_ds),N,replace=False)
fig,axes=plt.subplots(N,4,figsize=(18,N*3.2),facecolor='#06090f')
fig.suptitle('NeuroSAR Advanced — Test Predictions',color='white',fontsize=13,fontweight='bold')
for ci,t in enumerate(['SAR Input VV','Ground Truth','Prediction','Overlay']):
    axes[0,ci].set_title(t,color='#aaccff',fontsize=10,pad=6)
with torch.no_grad():
    for ri,idx in enumerate(indices):
        samp=test_ds[idx]; image=samp['image'].unsqueeze(0).to(DEVICE)
        gt=samp['mask'].numpy(); img_np=samp['image'][0].numpy()
        with autocast(enabled=(DEVICE.type=='cuda')): out=ema.shadow(image)
        pred=out['logits'].argmax(1).squeeze().cpu().numpy()
        for ci,(data,cmap) in enumerate([(img_np,'gray'),(gt,'custom'),(pred,'custom'),(img_np,'gray')]):
            ax=axes[ri,ci]; ax.set_facecolor('#0d1520')
            if cmap=='custom': ax.imshow(data,cmap=CLASS_CMAP,norm=CLASS_NORM,interpolation='nearest')
            else: ax.imshow(data,cmap=cmap,vmin=0,vmax=1)
            if ci==3:
                ov=np.zeros((*pred.shape,4))
                for c in range(1,NUM_CLASSES):
                    if (pred==c).any():
                        rgb=mcolors.to_rgb(CLASS_COLORS[c]); ov[pred==c]=(*rgb,0.65)
                ax.imshow(ov)
            ax.axis('off')
        axes[ri,0].set_ylabel(f'#{idx}',color='#aaccff',fontsize=9,rotation=0,labelpad=30,va='center')
patches=[mpatches.Patch(color=CLASS_COLORS[i],label=CLASS_NAMES[i]) for i in range(NUM_CLASSES)]
fig.legend(handles=patches,loc='lower center',ncol=4,fontsize=10,
           facecolor='#0d1520',edgecolor='#334',labelcolor='white',bbox_to_anchor=(0.5,0.01))
plt.tight_layout(rect=[0,0.04,1,0.97])
plt.savefig(RESULTS_DIR/'04_predictions.png',dpi=130,bbox_inches='tight',facecolor='#06090f')
plt.show(); print('Saved 04_predictions.png')

## Cell 13 — 8-Fold Test-Time Augmentation (TTA)
Averages predictions over 8 geometric augmentations for higher accuracy.

In [ ]:
def tta_predict(model,img_tensor,n_folds=8):
    model.eval()
    tfs=[lambda x:x,lambda x:torch.flip(x,[3]),lambda x:torch.flip(x,[2]),
         lambda x:torch.rot90(x,1,[2,3]),lambda x:torch.rot90(x,2,[2,3]),
         lambda x:torch.rot90(x,3,[2,3]),
         lambda x:torch.flip(torch.rot90(x,1,[2,3]),[3]),
         lambda x:torch.flip(torch.rot90(x,2,[2,3]),[2])]
    inv=[lambda x:x,lambda x:torch.flip(x,[3]),lambda x:torch.flip(x,[2]),
         lambda x:torch.rot90(x,3,[2,3]),lambda x:torch.rot90(x,2,[2,3]),
         lambda x:torch.rot90(x,1,[2,3]),
         lambda x:torch.flip(torch.rot90(x,3,[2,3]),[3]),
         lambda x:torch.flip(torch.rot90(x,2,[2,3]),[2])]
    preds=[]
    with torch.no_grad():
        for tf,iv in zip(tfs[:n_folds],inv[:n_folds]):
            with autocast(enabled=(DEVICE.type=='cuda')): out=model(tf(img_tensor))
            preds.append(iv(F.softmax(out['logits'],1)))
    return torch.stack(preds).mean(0)

test_ds=test_loader.dataset
fig,axes=plt.subplots(4,4,figsize=(20,20),facecolor='#06090f')
fig.suptitle('TTA — Single Pass vs 8-Fold Average',color='white',fontsize=13,fontweight='bold')
for ci,t in enumerate(['SAR input','GT mask','Single pass','8-fold TTA']): axes[0,ci].set_title(t,color='#aaccff',fontsize=10,pad=5)
ema.shadow.eval()
for ri in range(4):
    idx=np.random.randint(len(test_ds)); samp=test_ds[idx]
    img_t=samp['image'].unsqueeze(0).to(DEVICE); gt=samp['mask'].numpy(); img_np=samp['image'][0].numpy()
    with torch.no_grad(),autocast(enabled=(DEVICE.type=='cuda')):
        single=F.softmax(ema.shadow(img_t)['logits'],1).squeeze(0).cpu().numpy()
    tta=tta_predict(ema.shadow,img_t,n_folds=CFG['tta_folds']).squeeze(0).cpu().numpy()
    sm=single.argmax(0).astype(np.uint8); tm=tta.argmax(0).astype(np.uint8)
    for ci,(data,cmap) in enumerate([(img_np,'gray'),(gt,'custom'),(sm,'custom'),(tm,'custom')]):
        ax=axes[ri,ci]; ax.set_facecolor('#0d1520')
        if cmap=='custom': ax.imshow(data,cmap=CLASS_CMAP,norm=CLASS_NORM,interpolation='nearest')
        else: ax.imshow(data,cmap=cmap,vmin=0,vmax=1)
        ax.axis('off')
plt.tight_layout()
plt.savefig(RESULTS_DIR/'05_tta_comparison.png',dpi=120,bbox_inches='tight',facecolor='#06090f')
plt.show(); print('TTA complete — saved 05_tta_comparison.png')

## Cell 14 — GradCAM Explainability
Visualises what the ResNet-50 encoder attends to in each SAR tile.

In [ ]:
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    target_layer=model.enc.layer4[-1]
    cam=GradCAM(model=model,target_layers=[target_layer])
    test_ds=test_loader.dataset
    fig,axes=plt.subplots(3,4,figsize=(20,15),facecolor='#06090f')
    fig.suptitle('GradCAM — What the Model Focuses On',color='white',fontsize=13,fontweight='bold')
    for ci,t in enumerate(['SAR input','GT mask','Prediction','GradCAM']): axes[0,ci].set_title(t,color='#aaccff',fontsize=10)
    model.eval()
    for ri in range(3):
        idx=np.random.randint(len(test_ds)); samp=test_ds[idx]
        img_t=samp['image'].unsqueeze(0).to(DEVICE); gt=samp['mask'].numpy(); img_np=samp['image'][0].numpy()
        gc=cam(input_tensor=img_t)[0]
        with torch.no_grad(),autocast(enabled=(DEVICE.type=='cuda')):
            pred=model(img_t)['logits'].argmax(1).squeeze().cpu().numpy()
        cam_img=show_cam_on_image(np.stack([img_np]*3,axis=-1),gc,use_rgb=True)
        for ci,(data,cmap) in enumerate([(img_np,'gray'),(gt,'custom'),(pred,'custom'),(cam_img,'rgb')]):
            ax=axes[ri,ci]; ax.set_facecolor('#0d1520')
            if cmap=='custom': ax.imshow(data,cmap=CLASS_CMAP,norm=CLASS_NORM,interpolation='nearest')
            elif cmap=='rgb': ax.imshow(data)
            else: ax.imshow(data,cmap=cmap,vmin=0,vmax=1)
            ax.axis('off')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR/'06_gradcam.png',dpi=120,bbox_inches='tight',facecolor='#06090f')
    plt.show(); print('GradCAM saved — 06_gradcam.png')
except Exception as e: print(f'GradCAM skipped: {e}')

## Cell 15 — Train Advanced Siamese U-Net (Change Detection)

In [ ]:
class SiameseUNet(nn.Module):
    def __init__(self,in_channels=2,f=64,dropout=0.2):
        super().__init__()
        self.e1=nn.Sequential(ConvBNAct(in_channels,f),ConvBNAct(f,f))
        self.e2=nn.Sequential(nn.MaxPool2d(2),ConvBNAct(f,f*2),ConvBNAct(f*2,f*2))
        self.e3=nn.Sequential(nn.MaxPool2d(2),ConvBNAct(f*2,f*4),ConvBNAct(f*4,f*4))
        self.e4=nn.Sequential(nn.MaxPool2d(2),ConvBNAct(f*4,f*8),ConvBNAct(f*8,f*8))
        self.bot=nn.Sequential(nn.MaxPool2d(2),ConvBNAct(f*8,f*16),ASPP(f*16,f*16,rates=(1,6,12,18)))
        self.fb=ConvBNAct(f*32,f*16); self.f4=ConvBNAct(f*16,f*8)
        self.f3=ConvBNAct(f*8,f*4); self.f2=ConvBNAct(f*4,f*2); self.f1=ConvBNAct(f*2,f)
        self.u4=nn.ConvTranspose2d(f*16,f*8,2,stride=2)
        self.d4=nn.Sequential(ConvBNAct(f*16,f*8),ConvBNAct(f*8,f*8))
        self.u3=nn.ConvTranspose2d(f*8,f*4,2,stride=2)
        self.d3=nn.Sequential(ConvBNAct(f*8,f*4),ConvBNAct(f*4,f*4))
        self.u2=nn.ConvTranspose2d(f*4,f*2,2,stride=2)
        self.d2=nn.Sequential(ConvBNAct(f*4,f*2),ConvBNAct(f*2,f*2))
        self.u1=nn.ConvTranspose2d(f*2,f,2,stride=2)
        self.d1=nn.Sequential(ConvBNAct(f*2,f),ConvBNAct(f,f))
        self.head=nn.Sequential(ConvBNAct(f,f),nn.Dropout2d(dropout),nn.Conv2d(f,1,1))
    def encode(self,x):
        s1=self.e1(x); s2=self.e2(s1); s3=self.e3(s2); s4=self.e4(s3)
        return self.bot(s4),[s1,s2,s3,s4]
    def forward(self,bef,aft):
        bf,bs=self.encode(bef); af,as_=self.encode(aft)
        xb=self.fb(torch.cat([bf,af],1))
        s4=self.f4(torch.cat([bs[3],as_[3]],1)); s3=self.f3(torch.cat([bs[2],as_[2]],1))
        s2=self.f2(torch.cat([bs[1],as_[1]],1)); s1=self.f1(torch.cat([bs[0],as_[0]],1))
        d4=self.d4(torch.cat([self.u4(xb),s4],1)); d3=self.d3(torch.cat([self.u3(d4),s3],1))
        d2=self.d2(torch.cat([self.u2(d3),s2],1)); d1=self.d1(torch.cat([self.u1(d2),s1],1))
        return {'logits':self.head(d1)}

CD={'epochs':50,'batch_size':8,'lr_max':2e-3,'patience':15,'base_filters':64,'dropout':0.2}
cd_model=SiameseUNet(in_channels=CFG['in_channels'],f=CD['base_filters'],dropout=CD['dropout']).to(DEVICE)
cd_ema=EMA(cd_model,decay=0.9995)
cd_optim=optim.AdamW(cd_model.parameters(),lr=CD['lr_max'],weight_decay=1e-4)
cd_trl,cd_vl,cd_tel=get_loaders(DATA_DIR,{**CFG,**CD},'change')
cd_sched=OneCycleLR(cd_optim,max_lr=CD['lr_max'],steps_per_epoch=len(cd_trl),epochs=CD['epochs'],
                     pct_start=0.1,div_factor=25,final_div_factor=1e4)
cd_scaler=GradScaler(enabled=(DEVICE.type=='cuda'))
print(f'Siamese U-Net: {sum(p.numel() for p in cd_model.parameters())/1e6:.1f}M params')

cd_best=0.0; cd_pat=0; cd_best_path=EXP_DIR/'cd_best.pth'
for epoch in range(1,CD['epochs']+1):
    cd_model.train(); trl=0.0; trm=CDMetrics()
    for batch in tqdm(cd_trl,desc=f'CD Ep {epoch:3d}',leave=False):
        bef=batch['before'].to(DEVICE); aft=batch['after'].to(DEVICE); tgt=batch['change'].to(DEVICE)
        cd_optim.zero_grad(set_to_none=True)
        with autocast(enabled=(DEVICE.type=='cuda')):
            out=cd_model(bef,aft); loss=bfd_loss(out['logits'],tgt)
        cd_scaler.scale(loss).backward()
        cd_scaler.unscale_(cd_optim); nn.utils.clip_grad_norm_(cd_model.parameters(),1.0)
        cd_scaler.step(cd_optim); cd_scaler.update(); cd_sched.step(); cd_ema.update(cd_model)
        trl+=loss.item(); trm.update(out['logits'],tgt)
    cd_ema.shadow.eval(); vam=CDMetrics()
    with torch.no_grad():
        for batch in cd_vl:
            bef=batch['before'].to(DEVICE); aft=batch['after'].to(DEVICE); tgt=batch['change'].to(DEVICE)
            with autocast(enabled=(DEVICE.type=='cuda')): out=cd_ema.shadow(bef,aft)
            vam.update(out['logits'],tgt)
    vr=vam.compute(); star=''
    if vr['f1']>cd_best:
        cd_best=vr['f1']; cd_pat=0
        torch.save({'epoch':epoch,'state':cd_ema.shadow.state_dict(),'score':cd_best},cd_best_path); star=' BEST'
    else: cd_pat+=1
    print(f'Ep {epoch:3d} | F1={vr["f1"]:.4f}  IoU={vr["iou"]:.4f}  FAR={vr["far"]:.4f}  MAR={vr["mar"]:.4f}{star}')
    if cd_pat>=CD['patience']: print('Early stop'); break
print(f'CD done! Best F1 = {cd_best:.4f}')

## Cell 16 — Change Detection Results

In [ ]:
cd_ckpt=torch.load(cd_best_path,map_location=DEVICE)
cd_ema.shadow.load_state_dict(cd_ckpt['state']); cd_ema.shadow.eval()
print(f'Loaded CD EMA — epoch {cd_ckpt["epoch"]}  F1={cd_ckpt["score"]:.4f}')

cd_tm=CDMetrics()
with torch.no_grad():
    for batch in tqdm(cd_tel,desc='CD Test'):
        bef=batch['before'].to(DEVICE); aft=batch['after'].to(DEVICE); tgt=batch['change'].to(DEVICE)
        with autocast(enabled=(DEVICE.type=='cuda')): out=cd_ema.shadow(bef,aft)
        cd_tm.update(out['logits'],tgt)
cd_r=cd_tm.compute()
print('\n====== CHANGE DETECTION TEST RESULTS ======')
for k,v in cd_r.items(): print(f'  {k:15s}: {v:.4f}')

cd_ds=cd_tel.dataset
fig,axes=plt.subplots(4,5,figsize=(22,17),facecolor='#06090f')
fig.suptitle('Change Detection Results',color='white',fontsize=13,fontweight='bold')
for ci,t in enumerate(['Before','After','GT Change','Predicted','Difference']): axes[0,ci].set_title(t,color='#aaccff',fontsize=9,pad=5)
with torch.no_grad():
    for ri in range(4):
        samp=cd_ds[np.random.randint(len(cd_ds))]
        bt=samp['before'].unsqueeze(0).to(DEVICE); at_=samp['after'].unsqueeze(0).to(DEVICE)
        gt_c=samp['change'].squeeze().numpy()
        bn=samp['before'][0].numpy(); an=samp['after'][0].numpy()
        with autocast(enabled=(DEVICE.type=='cuda')): out=cd_ema.shadow(bt,at_)
        pp=torch.sigmoid(out['logits']).squeeze().cpu().numpy(); pb=(pp>0.5).astype(np.uint8)
        diff=np.abs(an.astype(float)-bn.astype(float))
        for ci,(data,cmap) in enumerate([(bn,'gray'),(an,'gray'),(gt_c,'hot'),(None,None),(diff,'inferno')]):
            ax=axes[ri,ci]; ax.set_facecolor('#0d1520')
            if ci==3:
                ax.imshow(an,cmap='gray',vmin=0,vmax=1)
                cov=np.zeros((*pb.shape,4)); cov[pb==1]=(1,0.2,0.2,0.75); cov[gt_c==1]=(0.2,1,0.2,0.3); ax.imshow(cov)
            elif cmap=='inferno': ax.imshow(data,cmap=cmap,vmin=0,vmax=data.max())
            elif cmap=='hot': ax.imshow(data,cmap=cmap,vmin=0,vmax=1)
            else: ax.imshow(data,cmap=cmap,vmin=0,vmax=1)
            ax.axis('off')
fig.text(0.5,0.01,'Red=predicted change   Green=ground truth',ha='center',color='#aaccff',fontsize=9)
plt.tight_layout(rect=[0,0.03,1,0.96])
plt.savefig(RESULTS_DIR/'07_change_detection.png',dpi=120,bbox_inches='tight',facecolor='#06090f')
plt.show(); print('Saved 07_change_detection.png')

## Cell 17 — Save to Google Drive & Download

In [ ]:
from google.colab import drive, files as colab_files
drive.mount('/content/drive')
GDRIVE=Path('/content/drive/MyDrive/NeuroSAR_Advanced'); GDRIVE.mkdir(parents=True,exist_ok=True)

shutil.copy(best_path,    RESULTS_DIR/'seg_model_best.pth')
shutil.copy(ema_path,     RESULTS_DIR/'seg_model_ema.pth')
shutil.copy(cd_best_path, RESULTS_DIR/'cd_model_ema.pth')

summary={'segmentation':results,'change_detection':cd_r,'cfg':CFG,'cd_cfg':CD,
          'best_seg_mIoU_targets':float(best_score),'best_cd_f1':float(cd_best)}
with open(RESULTS_DIR/'summary.json','w') as f: json.dump(summary,f,indent=2,default=str)

zip_path='/content/NeuroSAR_Advanced.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for f in RESULTS_DIR.iterdir(): zf.write(f,f.name)

shutil.copy(zip_path,GDRIVE/'NeuroSAR_Advanced.zip')
shutil.copy(ema_path,GDRIVE/'seg_model_ema.pth')
shutil.copy(cd_best_path,GDRIVE/'cd_model_ema.pth')

print(f'Saved to Google Drive: {GDRIVE}')
print('\n====== FINAL SUMMARY ======')
print(f'  Seg mIoU (targets): {best_score:.4f}')
print(f'  CD  F1            : {cd_best:.4f}')
print(f'  Encoder           : {CFG["encoder"]} pretrained ImageNet')
print(f'  Loss              : Focal + Lovász + Dice + DeepSup')
print(f'  Scheduler         : OneCycleLR superconvergence')
print(f'  EMA decay         : {CFG["ema_decay"]}')
print(f'  TTA folds         : {CFG["tta_folds"]}')
print(f'  Dual-pol          : VV+VH ({CFG["in_channels"]} channels)')
try: colab_files.download(zip_path); print('Download started')
except: print(f'Zip at: {zip_path}')

## Cell 18 — Real Sentinel-1 Data Guide

In [ ]:
guide = '''
STEP 1 — Register free at Copernicus Data Space
  https://dataspace.copernicus.eu/

STEP 2 — Download Sentinel-1 IW GRDH scenes
  pip install sentinelsat rasterio

  from sentinelsat import SentinelAPI, geojson_to_wkt
  api = SentinelAPI('user','pass','https://apihub.copernicus.eu/apihub')
  fp  = geojson_to_wkt({'type':'Polygon','coordinates':[[[30,45],[35,45],[35,50],[30,50],[30,45]]]})
  products = api.query(fp,date=('20230101','20231231'),platformname='Sentinel-1',
                       producttype='GRD',sensoroperationalmode='IW',polarisationmode='VV VH')
  api.download_all(products, directory_path='./raw')

STEP 3 — Preprocess with ESA SNAP (free)
  Download: https://step.esa.int/main/download/snap-download/
  Chain: Apply Orbit -> Thermal Noise Removal -> Calibration sigma0 dB
         -> Range-Doppler TC 10m -> Export GeoTIFF

STEP 4 — Drop-in replacement for SARSegDataset
  import rasterio, rasterio.windows as rw
  class RealSentinel1Dataset(Dataset):
    def __init__(self, tif_paths, label_paths, tile_size=256, stride=128):
        self.patches=[]
        for tp,lp in zip(tif_paths,label_paths):
            with rasterio.open(tp) as src: H,W=src.height,src.width
            for r in range(0,H-tile_size+1,stride):
                for c in range(0,W-tile_size+1,stride):
                    self.patches.append((tp,lp,r,c))
        self.ts=tile_size
    def __len__(self): return len(self.patches)
    def __getitem__(self,idx):
        tp,lp,r,c=self.patches[idx]; ts=self.ts
        with rasterio.open(tp) as src:
            vv=src.read(1,window=rw.Window(c,r,ts,ts)).astype(np.float32)
            vh=src.read(2,window=rw.Window(c,r,ts,ts)).astype(np.float32)
        vv=np.clip((vv+30)/35.0,0,1); vh=np.clip((vh+30)/35.0,0,1)
        image=np.stack([vv,vh],axis=-1)   # (256,256,2) dual-pol
        mask=np.load(lp)[r:r+ts,c:c+ts] if lp else np.zeros((ts,ts),dtype=np.int64)
        return {'image':torch.from_numpy(image.transpose(2,0,1)),
                'mask': torch.from_numpy(mask.astype(np.int64))}

  # Pass to existing training loop unchanged
  real_loader=DataLoader(RealSentinel1Dataset(tif_paths=[...],label_paths=[...]),
                          batch_size=8,shuffle=True,num_workers=2)

Annotation tools:
  QGIS     - free, opens GeoTIFF directly
  CVAT     - web-based semantic segmentation
  Roboflow - cloud annotation with auto-export
'''
print(guide)